<a href="https://colab.research.google.com/github/Madhaveffai/pytorch-learning/blob/main/week2/MNIST_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.308,))
])

train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True)

print(f"Train: {len(train_data)} | Test: {len(test_data)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 45.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.15MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.06MB/s]

Train: 60000 | Test: 10000


CNN

In [6]:
class MNISTModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(1, 8, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(8, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )
    self.classifier = nn.Sequential(
        nn.Linear(16 * 7 * 7, 64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 10)
    )
  def forward(self, x):
    x = self.features(x)
    x = x.view(x.size(0), -1)
    return self.classifier(x)

model = MNISTModel()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 52,138


Training Loop

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

def train_epoch(model, loader):
  model.train()
  total_loss, correct = 0, 0
  for X, y in loader:
    pred = model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    correct += (pred.argmax(1) == y).sum().item()
  return total_loss / len(loader), correct / len(loader.dataset)

def eval_epoch(model, loader):
  model.eval()
  total_loss, correct = 0, 0
  with torch.no_grad():
    for X, y in loader:
      pred = model(X)
      total_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).sum().item()
  return total_loss / len(loader), correct / len(loader.dataset)

Train for 10 epochs

In [12]:
for epoch in range(10):
  train_loss, train_acc = train_epoch(model, train_loader)
  test_loss, test_acc = eval_epoch(model, test_loader)
  print(f"Epoch {epoch+1:2d} |" f"Train Loss: {train_loss:.4f} acc: {train_acc} | "f"Test loss: {test_loss:.4f} acc: {test_acc:.4f}")

Epoch  1 |Train Loss: 0.0828 acc: 0.9749833333333333 | Test loss: 0.0501 acc: 0.9848
Epoch  2 |Train Loss: 0.0687 acc: 0.9796666666666667 | Test loss: 0.0433 acc: 0.9861
Epoch  3 |Train Loss: 0.0579 acc: 0.9817 | Test loss: 0.0415 acc: 0.9862
Epoch  4 |Train Loss: 0.0532 acc: 0.9834166666666667 | Test loss: 0.0421 acc: 0.9861
Epoch  5 |Train Loss: 0.0467 acc: 0.98545 | Test loss: 0.0382 acc: 0.9877
Epoch  6 |Train Loss: 0.0423 acc: 0.9866666666666667 | Test loss: 0.0385 acc: 0.9873
Epoch  7 |Train Loss: 0.0388 acc: 0.9874666666666667 | Test loss: 0.0327 acc: 0.9893
Epoch  8 |Train Loss: 0.0368 acc: 0.9878833333333333 | Test loss: 0.0384 acc: 0.9890
Epoch  9 |Train Loss: 0.0347 acc: 0.9887833333333333 | Test loss: 0.0360 acc: 0.9896
Epoch 10 |Train Loss: 0.0325 acc: 0.9892833333333333 | Test loss: 0.0420 acc: 0.9873


Saving the model

In [13]:
torch.save(model.state_dict(), "mnist_mode.pth")
print("Model saved!")

Model saved!
